# Speech Commands: exploración y representación de audio como imagen

**Objetivo:** preparar ejemplos etiquetados para una futura CNN que clasifique
`yes, no, left, right, up, down, on, off, stop, go` y **`unknown`**, que agrupa
las demás palabras del corpus. Son **11 clases**. Es un problema de clasificación
supervisada multiclase: cada grabación proporciona una entrada y su carpeta, la etiqueta.
Este notebook llega hasta la exportación y verificación de las entradas; el entrenamiento
será un paso posterior.

Ejecutar todas las celdas en orden con Python 3.11. La primera ejecución descarga el
dataset de Kaggle, que puede ocupar varios GB, y procesa todos los WAV seleccionados.
Los resultados incluyen tablas, gráficos, audio reproducible, imágenes PNG y tensores NumPy.
No se presupone el número de archivos ni su calidad: se calculan a partir de la descarga.

El corpus original es **Speech Commands v0.02**, de Pete Warden / Google,
distribuido bajo **CC BY 4.0**; consultar `README.md` y `LICENSE` en la descarga.
Referencia: [Speech Commands: A Dataset for Limited-Vocabulary Speech Recognition](https://arxiv.org/abs/1804.03209).

Fuentes: [dataset de Kaggle](https://www.kaggle.com/datasets/yashdogra/speech-commands),
[descargas versionadas con kagglehub](https://github.com/Kaggle/kagglehub#download-dataset),
[Speech Commands](https://www.tensorflow.org/datasets/catalog/speech_commands),
[espectrogramas Mel con librosa](https://librosa.org/doc/0.11.0/generated/librosa.feature.melspectrogram.html).


## 1. Preparar el entorno

Desde la raíz del repositorio: `python -m pip install -r ai/requirements.txt`.
Seleccionar ese entorno como kernel de Jupyter. La siguiente celda opcional instala
las mismas dependencias desde el notebook; descomentar si faltan paquetes.


In [ ]:
from pathlib import Path

# Funciona al abrir Jupyter desde la raíz, ai/ o ai/notebooks/.
AI_DIR = next(
    (p if p.name == "ai" else p / "ai"
     for p in [Path.cwd(), *Path.cwd().parents]
     if (p if p.name == "ai" else p / "ai").joinpath("requirements.txt").is_file()),
    None,
)
if AI_DIR is None:
    raise RuntimeError("Abra este notebook dentro del repositorio OnnxMobile.")
REQUIREMENTS = AI_DIR / "requirements.txt"
# %pip install -r {REQUIREMENTS}
print("Directorio de trabajo:", AI_DIR)


In [ ]:
import os
import sys
import json
import hashlib
import io
import platform
from concurrent.futures import ThreadPoolExecutor
from itertools import islice
from datetime import datetime, timezone
from importlib.metadata import version

# Mantener cachés dentro del proyecto, antes de importar las librerías.
os.environ.setdefault("KAGGLEHUB_CACHE", str(AI_DIR / "data" / "raw" / "kagglehub"))
os.environ.setdefault("MPLCONFIGDIR", str(AI_DIR / ".cache" / "matplotlib"))
os.environ.setdefault("NUMBA_CACHE_DIR", str(AI_DIR / ".cache" / "numba"))
import kagglehub
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import soundfile as sf
from PIL import Image
from IPython.display import Audio, Markdown, display
from tqdm.auto import tqdm
from threadpoolctl import threadpool_limits

# Matrices pequeñas: un hilo BLAS evita sobrecostes de coordinación por clip.
BLAS_LIMIT = threadpool_limits(limits=1, user_api="blas")

SEED = 42
COMMANDS = ["yes", "no", "left", "right", "up", "down", "on", "off", "stop", "go"]
CLASSES = COMMANDS + ["unknown"]
LABEL_TO_ID = {label: i for i, label in enumerate(CLASSES)}
WORKERS = 4  # E/S concurrente; usar 1 para ejecución secuencial.


def parallel_map(function, items):
    """Procesar lotes pequeños conservando el orden y limitando la memoria."""
    iterator = iter(items)
    with ThreadPoolExecutor(max_workers=WORKERS) as pool:
        while batch := list(islice(iterator, WORKERS * 8)):
            yield from pool.map(function, batch)
DATASET_HANDLE = "yashdogra/speech-commands"
# None: obtener la última versión en la primera ejecución y fijarla en dataset_lock.json.
# Para otra versión, indicar su número explícitamente.
DATASET_VERSION = None
# Opcional: ruta a una descarga existente; debe contener las carpetas de comandos.
LOCAL_DATASET = os.environ.get("SPEECH_COMMANDS_DATA_DIR")
OUTPUT_ROOT = Path(os.environ.get(
    "SPEECH_COMMANDS_OUTPUT_DIR", str(AI_DIR / "data" / "processed" / "speech_commands")
)).resolve()
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SAMPLE_RATE = 16_000
DURATION = 1.0
N_SAMPLES = int(SAMPLE_RATE * DURATION)
N_FFT = 512
WIN_LENGTH = 400  # 25 ms
HOP_LENGTH = 160  # 10 ms
N_MELS = 64
TOP_DB = 80.0
N_FRAMES = 1 + N_SAMPLES // HOP_LENGTH  # center=True: 101 columnas
PACKAGES = ["kagglehub", "numpy", "pandas", "scipy", "librosa", "numba",
            "soundfile", "matplotlib", "Pillow", "tqdm", "nbformat", "nbclient", "ipykernel", "threadpoolctl"]
ENVIRONMENT = {name: version(name) for name in PACKAGES}
display(pd.Series(ENVIRONMENT, name="Versión"))


## 2. Descargar e identificar la versión

La primera descarga usa `kagglehub.dataset_download("yashdogra/speech-commands")`.
Se guarda la versión resuelta y las siguientes ejecuciones la reutilizan: “latest”
por sí solo no garantiza reproducibilidad. Una ruta local evita la descarga;
en ambos casos se registran hashes SHA-256 de los WAV utilizados.
Si Kaggle pide autenticación, configurar las credenciales fuera del notebook
según su documentación, sin pegarlas en las celdas.


In [ ]:
lock_path = OUTPUT_ROOT / "dataset_lock.json"
if LOCAL_DATASET:
    dataset_path = Path(LOCAL_DATASET).expanduser().resolve()
    source_info = {"mode": "local", "path": str(dataset_path), "handle": DATASET_HANDLE}
else:
    selected_version = DATASET_VERSION
    if selected_version is None and lock_path.exists():
        previous = json.loads(lock_path.read_text(encoding="utf-8"))
        if previous.get("handle") == DATASET_HANDLE:
            selected_version = previous.get("version")
    handle = DATASET_HANDLE
    if selected_version is not None:
        handle += f"/versions/{int(selected_version)}"
    dataset_path = Path(kagglehub.dataset_download(handle)).resolve()
    # kagglehub almacena las descargas locales en .../versions/<n>/.
    resolved_version = selected_version
    if dataset_path.parent.name == "versions" and dataset_path.name.isdigit():
        resolved_version = int(dataset_path.name)
    if resolved_version is None:
        raise RuntimeError("No se pudo identificar la versión; configure DATASET_VERSION explícitamente.")
    source_info = {"mode": "kagglehub", "handle": DATASET_HANDLE,
                   "version": int(resolved_version), "path": str(dataset_path)}
    lock_path.write_text(json.dumps(source_info, indent=2), encoding="utf-8")
if not dataset_path.is_dir():
    raise FileNotFoundError(dataset_path)
print("Path to dataset files:", dataset_path)
display(source_info)


## 3. Inventario y asignación de las once clases

Se detecta la carpeta que contiene todos los comandos, incluso si Kaggle añade
directorios intermedios. Si hay varias raíces posibles, se exige elegir una con
`LOCAL_DATASET` para no mezclar copias o particiones accidentalmente.
Cada comando conserva su clase (IDs 0–9). Todas las otras palabras se asignan a
`unknown` (ID 10), sin submuestreo; `source_label` conserva la palabra original.
Las carpetas auxiliares cuyo nombre empieza por `_`, incluido `_background_noise_`,
no son palabras y se excluyen de esta categoría. Silencio y ruido necesitarían
un tratamiento propio en un reconocedor continuo.

`unknown` será mucho más grande que cada comando. Se conserva todo su contenido
válido para esta exploración; al entrenar se podrá ponderar la pérdida o muestrear
solo entrenamiento. Validación y prueba conservan su distribución real.


In [ ]:
candidates = sorted({
    wav.parent.parent for wav in dataset_path.rglob("*.wav")
    if wav.parent.name == COMMANDS[0]
    and all((wav.parent.parent / label).is_dir() for label in COMMANDS)
})
if len(candidates) != 1:
    raise RuntimeError(f"Se esperaba una raíz de audio; encontradas {len(candidates)}: {candidates}")
audio_root = candidates[0]
inventory = pd.DataFrame([
    {"label": folder.name, "n_files": len(list(folder.glob("*.wav"))),
     "selected": not folder.name.startswith("_"),
     "target_label": (folder.name if folder.name in COMMANDS else "unknown")
                     if not folder.name.startswith("_") else "excluded"}
    for folder in sorted(audio_root.iterdir()) if folder.is_dir()
]).query("n_files > 0")
display(inventory)
inventory.to_csv(OUTPUT_ROOT / "inventory.csv", index=False)
ax = inventory.set_index("label")["n_files"].plot.bar(
    figsize=(14, 4), color=["steelblue" if x in COMMANDS else "darkorange" if x == "unknown" else "lightgray"
           for x in inventory.target_label]
)
ax.set(title="Azul: comandos · naranja: unknown · gris: excluido", ylabel="Archivos WAV", xlabel="Etiqueta")
plt.tight_layout()
plt.show()
SOURCE_WORDS = inventory.loc[inventory.selected, "label"].tolist()
UNKNOWN_WORDS = sorted(set(SOURCE_WORDS) - set(COMMANDS))
assert UNKNOWN_WORDS, "No hay otras palabras para construir unknown."
SOURCE_TO_LABEL = {word: word if word in COMMANDS else "unknown" for word in SOURCE_WORDS}
selected_paths = sorted(p for word in SOURCE_WORDS for p in (audio_root / word).glob("*.wav"))
print("Palabras agrupadas en unknown:", UNKNOWN_WORDS)
display(inventory.groupby("target_label").n_files.sum().reindex(CLASSES).rename("Grabaciones por clase"))
assert selected_paths, "No se encontraron grabaciones."
print("Grabaciones seleccionadas:", len(selected_paths))


## 4. Auditar el audio antes de modificarlo

Se leen los WAV completos para detectar archivos dañados, longitudes cero,
valores no finitos, silencio digital exacto y grabaciones duplicadas. Se miden
frecuencia de muestreo, canales, duración, amplitud máxima, RMS y proporción de
muestras próximas a saturación (`|x| >= 0.999`). RMS bajo o posible saturación
son señales para inspeccionar, no causas automáticas de descarte.

Se excluyen archivos inválidos y silencio exacto; se conserva una copia de cada
audio idéntico decodificado con la misma palabra de origen. Duplicados con palabras
de origen contradictorias (incluso dentro de `unknown`) se excluyen por completo. El informe conserva cada exclusión.
El hablante se obtiene del prefijo anterior a `_nohash_`, siguiendo los nombres
de Speech Commands; nombres incompatibles provocan un error explícito.


In [ ]:
def audit_audio(path):
    source_label = path.parent.name
    label = SOURCE_TO_LABEL[source_label]
    record = {"relative_path": path.relative_to(audio_root).as_posix(),
              "source_label": source_label, "label": label,
              "speaker": path.stem.split("_nohash_")[0],
              "label_id": LABEL_TO_ID[label], "error": ""}
    if "_nohash_" not in path.stem:
        raise ValueError(f"No se puede identificar el hablante: {path.name}")
    try:
        raw = path.read_bytes()
        record["file_sha256"] = hashlib.sha256(raw).hexdigest()
        signal, sr = sf.read(io.BytesIO(raw), dtype="float32", always_2d=True)
        if len(signal) == 0 or not np.isfinite(signal).all():
            raise ValueError("Audio vacío o con valores no finitos")
        peak = float(np.max(np.abs(signal)))
        record.update(sample_rate=sr, channels=signal.shape[1], frames=len(signal),
                      duration_s=len(signal) / sr, peak=peak,
                      rms=float(np.sqrt(np.mean(signal.astype(np.float64) ** 2))),
                      clipping_fraction=float(np.mean(np.abs(signal) >= 0.999)))
        payload = f"{sr}:{signal.shape}:".encode() + signal.astype("<f4").tobytes()
        record["audio_sha256"] = hashlib.sha256(payload).hexdigest()
        if peak == 0:
            record["error"] = "silencio_digital"
    except (RuntimeError, ValueError, OSError) as exc:
        record["error"] = f"{type(exc).__name__}: {exc}"
    return record

audit = pd.DataFrame(tqdm(parallel_map(audit_audio, selected_paths),
                          total=len(selected_paths), desc="Auditoría de WAV"))
audit["exclusion_reason"] = audit["error"]
valid = audit["error"].eq("")
if not valid.any():
    audit.to_csv(OUTPUT_ROOT / "audio_audit.csv", index=False)
    raise RuntimeError("No quedan grabaciones válidas; consulte audio_audit.csv.")
label_counts = audit.loc[valid].groupby("audio_sha256")["source_label"].nunique()
conflicting = label_counts[label_counts > 1].index
audit.loc[valid & audit.audio_sha256.isin(conflicting), "exclusion_reason"] = "etiquetas_contradictorias"
eligible = audit.exclusion_reason.eq("")
duplicate_indices = audit.loc[eligible].loc[
    audit.loc[eligible].duplicated("audio_sha256", keep="first")
].index
audit.loc[duplicate_indices, "exclusion_reason"] = "audio_duplicado"
audit.to_csv(OUTPUT_ROOT / "audio_audit.csv", index=False)
clean = audit.loc[audit.exclusion_reason.eq("")].copy().reset_index(drop=True)
assert set(clean.label) == set(CLASSES), "Falta alguna clase después de la limpieza."
display(audit.exclusion_reason.replace("", "conservado").value_counts().rename("Archivos"))
display(audit.loc[valid].groupby("label").agg(
    archivos=("relative_path", "size"), hablantes=("speaker", "nunique"),
    duracion_min=("duration_s", "min"), duracion_mediana=("duration_s", "median"),
    duracion_max=("duration_s", "max"), rms_mediana=("rms", "median"),
).reindex(CLASSES))
display(audit.loc[valid].groupby(["sample_rate", "channels"]).size().rename("Archivos"))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
clean.label.value_counts().reindex(CLASSES).plot.bar(ax=axes[0], title="Clases después de limpieza")
clean.duration_s.hist(ax=axes[1], bins=40)
axes[1].set(title="Duraciones originales", xlabel="Segundos", ylabel="Archivos")
axes[2].hist(20 * np.log10(np.maximum(clean.rms, 1e-10)), bins=50)
axes[2].set(title="Nivel RMS original", xlabel="dB relativos a amplitud 1", ylabel="Archivos")
plt.tight_layout()
plt.show()
print("Audios con muestras próximas a saturación:", int((clean.clipping_fraction > 0).sum()))


## 5. Separar entrenamiento, validación y prueba por hablante

Se prefieren `validation_list.txt` y `testing_list.txt` originales; entrenamiento
es su complemento. Si no existen, un hash SHA-256 del hablante y la semilla
asigna aproximadamente 80 % / 10 % / 10 %. Estas proporciones no son exactas
por clase. Nunca se reparten grabaciones de un mismo hablante al azar.
Se verifica que no se compartan hablantes ni contenido entre particiones y que
cada partición contenga las once clases. La prueba queda reservada para la
evaluación final; la exploración visual siguiente usa solo entrenamiento.


In [ ]:
def locate_list(filename):
    paths = sorted(dataset_path.rglob(filename))
    if len(paths) > 1:
        raise RuntimeError(f"Hay varias listas {filename}: {paths}")
    return paths[0] if paths else None

validation_file = locate_list("validation_list.txt")
testing_file = locate_list("testing_list.txt")
if bool(validation_file) != bool(testing_file):
    raise RuntimeError("Solo existe una de las dos listas oficiales de partición.")

split_list_hashes = {}
if validation_file and testing_file:
    def read_list(path):
        split_list_hashes[path.name] = hashlib.sha256(path.read_bytes()).hexdigest()
        return {line.strip().replace(chr(92), "/").removeprefix("./")
                for line in path.read_text(encoding="utf-8").splitlines() if line.strip()}
    validation_set, testing_set = read_list(validation_file), read_list(testing_file)
    assert validation_set.isdisjoint(testing_set), "Listas oficiales solapadas."
    known_paths = set(audit.relative_path)
    for name, entries in [("validation", validation_set), ("test", testing_set)]:
        selected_entries = {p for p in entries if p.split("/")[0] in SOURCE_WORDS}
        assert selected_entries and selected_entries <= known_paths, f"Lista {name} incompatible con los WAV."
    clean["split"] = clean.relative_path.map(
        lambda p: "validation" if p in validation_set else "test" if p in testing_set else "train"
    )
    split_method = "official_lists"
else:
    def speaker_split(speaker):
        digest = hashlib.sha256(f"{SEED}:{speaker}".encode()).digest()
        score = int.from_bytes(digest[:8], "big") / 2**64
        return "test" if score < 0.1 else "validation" if score < 0.2 else "train"
    clean["split"] = clean.speaker.map(speaker_split)
    split_method = "speaker_sha256_80_10_10"

assert clean.groupby("speaker").split.nunique().max() == 1, "Fuga de hablantes."
assert clean.groupby("audio_sha256").split.nunique().max() == 1, "Fuga de contenido."
split_counts = pd.crosstab(clean.label, clean.split).reindex(
    index=CLASSES, columns=["train", "validation", "test"], fill_value=0
)
assert (split_counts > 0).all().all(), "Alguna partición carece de clases; revisar dataset y partición."
display(split_counts)
display(clean.groupby("split").agg(archivos=("label", "size"), hablantes=("speaker", "nunique")))
split_counts.to_csv(OUTPUT_ROOT / "split_counts.csv")
unknown_counts = pd.crosstab(
    clean.loc[clean.label.eq("unknown"), "source_label"],
    clean.loc[clean.label.eq("unknown"), "split"],
).reindex(index=UNKNOWN_WORDS, columns=["train", "validation", "test"], fill_value=0)
display(unknown_counts)
unknown_counts.to_csv(OUTPUT_ROOT / "unknown_word_counts.csv")
split_counts.plot.bar(stacked=True, figsize=(11, 4), title=f"Distribución por partición: {split_method}")
plt.ylabel("Grabaciones")
plt.tight_layout()
plt.show()


## 6. Audio → espectrograma log-Mel → imagen

1. Decodificar a `float32`, promediar canales y remuestrear a **16 kHz**.
2. Ajustar a **1 segundo**: rellenar al final con ceros si falta audio y recortar
   desde el centro si sobra. No se elimina silencio automáticamente ni se aumenta volumen.
3. Calcular potencia STFT con ventana Hann de **25 ms**, salto de **10 ms** y FFT
   de **512** puntos; aplicar **64 filtros Mel**, de 0 a 8 kHz, escala Slaney.
4. Convertir potencia a dB respecto del máximo de cada grabación, limitar a
   `[-80, 0]` dB y mapear linealmente a `[0, 1]`. Esto reduce la dependencia del
   volumen absoluto; no usa estadísticas de validación ni prueba. Silencio exacto
   produce una imagen negra. No se aplica normalización min-max por imagen adicional.
5. Guardar **64 × 101** píxeles en un canal: filas = bandas Mel de baja a alta,
   columnas = tiempo. La CNN recibirá `(N, 64, 101, 1)` en formato NHWC.

`center=True` añade padding a la STFT y da `1 + 16000 // 160 = 101` columnas.
No se fuerza una imagen cuadrada ni se añaden ejes, texto o colores a los datos
de entrenamiento. Los gráficos con leyendas son únicamente para exploración.
PNG cuantiza a 8 bits; los tensores `.npy` conservan `float32`.


In [ ]:
def load_fixed_audio(path):
    signal, sr = sf.read(path, dtype="float32", always_2d=True)
    if len(signal) == 0 or not np.isfinite(signal).all():
        raise ValueError(f"Audio inválido: {path}")
    mono = signal.mean(axis=1)
    if sr != SAMPLE_RATE:
        mono = librosa.resample(mono, orig_sr=sr, target_sr=SAMPLE_RATE, res_type="soxr_hq")
    if len(mono) > N_SAMPLES:
        start = (len(mono) - N_SAMPLES) // 2
        mono = mono[start:start + N_SAMPLES]
    else:
        mono = np.pad(mono, (0, N_SAMPLES - len(mono)))
    return mono.astype(np.float32)


def audio_to_image(signal):
    if signal.shape != (N_SAMPLES,) or not np.isfinite(signal).all():
        raise ValueError("Se requiere audio mono finito de exactamente 16000 muestras.")
    mel = librosa.feature.melspectrogram(
        y=signal, sr=SAMPLE_RATE, n_fft=N_FFT, win_length=WIN_LENGTH,
        hop_length=HOP_LENGTH, window="hann", center=True, pad_mode="constant",
        power=2.0, n_mels=N_MELS, fmin=0.0, fmax=SAMPLE_RATE / 2,
        htk=False, norm="slaney",
    )
    if float(mel.max()) <= 1e-10:
        db = np.full_like(mel, -TOP_DB)
    else:
        db = librosa.power_to_db(mel, ref=np.max, amin=1e-10, top_db=TOP_DB)
    image = np.clip((db + TOP_DB) / TOP_DB, 0.0, 1.0).astype(np.float32)
    assert image.shape == (N_MELS, N_FRAMES) and np.isfinite(image).all()
    return image


training = clean.loc[clean.split.eq("train")]
examples = pd.concat([
    training.loc[training.label.eq(label)].sample(n=1, random_state=SEED)
    for label in CLASSES
])
print("Forma de la imagen:", (N_MELS, N_FRAMES))
print("Audios que se rellenarán:", int((clean.duration_s < DURATION).sum()))
print("Audios que se recortarán:", int((clean.duration_s > DURATION).sum()))
fig, axes = plt.subplots(len(CLASSES), 2, figsize=(13, 25), constrained_layout=True)
for row_index, row in enumerate(examples.itertuples(index=False)):
    signal = load_fixed_audio(audio_root / row.relative_path)
    image = audio_to_image(signal)
    axes[row_index, 0].plot(np.arange(N_SAMPLES) / SAMPLE_RATE, signal, linewidth=0.6)
    axes[row_index, 0].set(title=f"{row.label} ({row.source_label}) · forma de onda", xlabel="Tiempo (s)", ylabel="Amplitud")
    plot = librosa.display.specshow(
        image * TOP_DB - TOP_DB, sr=SAMPLE_RATE, hop_length=HOP_LENGTH,
        x_axis="time", y_axis="mel", fmin=0, fmax=SAMPLE_RATE / 2,
        ax=axes[row_index, 1], vmin=-TOP_DB, vmax=0, cmap="magma",
    )
    axes[row_index, 1].set(title=f"{row.label} · log-Mel")
fig.colorbar(plot, ax=axes[:, 1], label="dB respecto del máximo del clip", shrink=0.5)
plt.show()
for row in examples.itertuples(index=False):
    display(Markdown(f"**{row.label}** — `{row.relative_path}`"))
    display(Audio(load_fixed_audio(audio_root / row.relative_path), rate=SAMPLE_RATE, normalize=False))


## 7. Exportar datos y trazabilidad

Cada ejecución crea una carpeta nueva. Se guarda una imagen PNG por grabación en
`images/<split>/<label>/`, además de tensores `X_<split>.npy`, etiquetas enteras
`y_<split>.npy` y un manifiesto que enlaza cada fila del tensor con el WAV y el PNG.
`open_memmap` escribe sin mantener todo el dataset en RAM. Los PNG incluyen la
palabra original en el nombre (`bed__<hablante>_nohash_0.png`), porque palabras
distintas pueden compartir el mismo nombre WAV dentro de `unknown`. La lectura,
transformación y verificación usan lotes de cuatro trabajadores y preservan
el orden; `WORKERS = 1` permite ejecutar el mismo procedimiento secuencialmente.

`preprocessing.json` conserva todos los parámetros, el orden de clases, la versión
del dataset y las versiones del entorno. `requirements-lock.txt` captura también
dependencias transitivas. La misma transformación deberá usarse al inferir en móvil.
`COMPLETED.json` solo aparece cuando las comprobaciones finales pasan: no entrenar
con una carpeta incompleta.


In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RUN_DIR = OUTPUT_ROOT / run_id
RUN_DIR.mkdir(parents=True, exist_ok=False)
config = {
    "seed": SEED, "source": source_info, "classes": CLASSES, "label_to_id": LABEL_TO_ID,
    "commands": COMMANDS, "unknown_words": UNKNOWN_WORDS, "source_to_label": SOURCE_TO_LABEL,
    "unknown_sampling": "all_valid_unique_audio", "workers": WORKERS,
    "duplicate_conflict_key": "source_label",
    "sample_rate": SAMPLE_RATE, "duration_s": DURATION, "n_samples": N_SAMPLES,
    "mono": "channel_mean", "resample": "soxr_hq", "short_audio": "zero_pad_end",
    "long_audio": "center_crop", "n_fft": N_FFT, "win_length": WIN_LENGTH,
    "hop_length": HOP_LENGTH, "window": "hann", "center": True, "pad_mode": "constant",
    "power": 2.0, "n_mels": N_MELS, "fmin": 0.0, "fmax": SAMPLE_RATE / 2,
    "htk": False, "mel_norm": "slaney", "db_ref": "per_clip_max", "amin": 1e-10,
    "top_db": TOP_DB, "normalization": "clip((db + top_db) / top_db, 0, 1)",
    "image_shape": [N_MELS, N_FRAMES, 1], "tensor_layout": "NHWC", "dtype": "float32",
    "png": "uint8 grayscale, row 0 = lowest Mel band, round(255 * image)",
    "split_method": split_method, "split_list_sha256": split_list_hashes,
    "python": sys.version, "platform": platform.platform(), "packages": ENVIRONMENT,
    "created_utc": run_id,
}
(RUN_DIR / "preprocessing.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
import subprocess
freeze = subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True)
(RUN_DIR / "requirements-lock.txt").write_text(freeze, encoding="utf-8")
inventory.to_csv(RUN_DIR / "inventory.csv", index=False)
audit.to_csv(RUN_DIR / "audio_audit.csv", index=False)
split_counts.to_csv(RUN_DIR / "split_counts.csv")
unknown_counts.to_csv(RUN_DIR / "unknown_word_counts.csv")


def export_image(row):
    source = audio_root / row.relative_path
    raw = source.read_bytes()
    if hashlib.sha256(raw).hexdigest() != row.file_sha256:
        raise RuntimeError(f"El WAV cambió desde la auditoría: {source}")
    image = audio_to_image(load_fixed_audio(io.BytesIO(raw)))
    image_relative = Path("images") / row.split / row.label / f"{row.source_label}__{source.stem}.png"
    Image.fromarray(np.rint(image * 255).astype(np.uint8)).save(RUN_DIR / image_relative)
    item = row._asdict()
    item["image_path"] = image_relative.as_posix()
    return image, item


manifest_rows = []
for split in ["train", "validation", "test"]:
    subset = clean.loc[clean.split.eq(split)].sort_values(["label_id", "relative_path"])
    tensors = np.lib.format.open_memmap(
        RUN_DIR / f"X_{split}.npy", mode="w+", dtype=np.float32,
        shape=(len(subset), N_MELS, N_FRAMES, 1),
    )
    labels = np.empty(len(subset), dtype=np.int64)
    for label in CLASSES:
        (RUN_DIR / "images" / split / label).mkdir(parents=True, exist_ok=True)
    results = parallel_map(export_image, subset.itertuples(index=False))
    for tensor_index, (image, item) in enumerate(tqdm(results, total=len(subset), desc=split)):
        tensors[tensor_index, :, :, 0] = image
        labels[tensor_index] = item["label_id"]
        item["tensor_index"] = tensor_index
        manifest_rows.append(item)
    tensors.flush()
    del tensors
    np.save(RUN_DIR / f"y_{split}.npy", labels, allow_pickle=False)

manifest = pd.DataFrame(manifest_rows)
assert manifest.image_path.is_unique, "Colisión de rutas PNG."
manifest.to_csv(RUN_DIR / "manifest.csv", index=False)
print("Exportación:", RUN_DIR)


## 8. Verificar los artefactos y mostrar cómo cargar una entrada de CNN

Se vuelven a leer todos los PNG y tensores para verificar forma, rango, etiquetas,
índices, ausencia de NaN y concordancia de la cuantización. Se comprueba un ejemplo
regenerado desde su WAV por partición. Estas verificaciones validan el procesamiento,
no la capacidad predictiva de un modelo todavía no entrenado.


In [ ]:
for split in ["train", "validation", "test"]:
    X = np.load(RUN_DIR / f"X_{split}.npy", mmap_mode="r", allow_pickle=False)
    y = np.load(RUN_DIR / f"y_{split}.npy", allow_pickle=False)
    rows = manifest.loc[manifest.split.eq(split)].sort_values("tensor_index")
    assert X.shape == (len(rows), N_MELS, N_FRAMES, 1) and X.dtype == np.float32
    assert y.dtype == np.int64 and np.array_equal(y, rows.label_id.to_numpy())
    assert np.array_equal(rows.tensor_index, np.arange(len(rows)))
    def verify_image(row):
        tensor = X[row.tensor_index, :, :, 0]
        assert np.isfinite(tensor).all() and 0 <= tensor.min() <= tensor.max() <= 1
        with Image.open(RUN_DIR / row.image_path) as png:
            assert png.mode == "L" and png.size == (N_FRAMES, N_MELS)
            pixels = np.asarray(png)
        assert np.array_equal(pixels, np.rint(tensor * 255).astype(np.uint8))
    checks = parallel_map(verify_image, rows.itertuples(index=False))
    for _ in tqdm(checks, total=len(rows), desc=f"Verificar {split}"):
        pass
    first = rows.iloc[0]
    regenerated = audio_to_image(load_fixed_audio(audio_root / first.relative_path))
    np.testing.assert_allclose(X[0, :, :, 0], regenerated, rtol=1e-6, atol=1e-7)
    print(split, X.shape, y.shape)
    del X

completion = {"counts": manifest.split.value_counts().to_dict(),
              "manifest_sha256": hashlib.sha256((RUN_DIR / "manifest.csv").read_bytes()).hexdigest()}
(RUN_DIR / "COMPLETED.json").write_text(json.dumps(completion, indent=2), encoding="utf-8")

# Ejemplo de lectura para entrenamiento posterior, sin cargar todos los datos.
X_train = np.load(RUN_DIR / "X_train.npy", mmap_mode="r", allow_pickle=False)
y_train = np.load(RUN_DIR / "y_train.npy", allow_pickle=False)
rng = np.random.default_rng(SEED)
indices = rng.choice(len(y_train), size=min(32, len(y_train)), replace=False)
batch_nhwc = np.asarray(X_train[indices])
batch_nchw = np.transpose(batch_nhwc, (0, 3, 1, 2))  # Conv2d en PyTorch
print("Lote NHWC:", batch_nhwc.shape, "| NCHW:", batch_nchw.shape)
print("Etiquetas:", y_train[indices])
print("Orden explícito de clases:", LABEL_TO_ID)
# Si se usa un cargador de carpetas, respetar este orden; no asumir orden alfabético.


## 9. Conclusiones calculadas y siguiente etapa

El resumen siguiente se calcula con los datos reales de esta ejecución. La
representación elegida conserva patrones de tiempo y frecuencia para una CNN.
Las diferencias visuales entre unos pocos ejemplos no demuestran separabilidad
ni permiten anticipar exactitud: eso requiere entrenar y evaluar.


In [ ]:
counts = clean.label.value_counts().reindex(CLASSES)
summary = f"""
- WAV seleccionados: **{len(audit):,}**; conservados: **{len(clean):,}**;
  excluidos: **{len(audit) - len(clean):,}** (detalle en `audio_audit.csv`).
- Clases: **{len(counts)}**; mínimo/máximo por clase: **{counts.min():,} / {counts.max():,}**;
  relación máximo/mínimo: **{counts.max() / counts.min():.2f}**.
- `unknown`: **{counts['unknown']:,}** grabaciones de **{len(UNKNOWN_WORDS)}** palabras;
  representa **{100 * counts['unknown'] / len(clean):.1f} %** de los ejemplos conservados.
- Hablantes conservados: **{clean.speaker.nunique():,}**, sin solapamiento entre particiones.
- Duraciones conservadas: **{clean.duration_s.min():.3f}–{clean.duration_s.max():.3f} s**.
- Imagen: **{N_MELS} × {N_FRAMES} × 1**, `float32` en `[0, 1]`; PNG en escala de grises.
- Artefactos verificados: `{RUN_DIR}`.

Siguiente etapa: entrenar una CNN con `train`, elegir arquitectura e hiperparámetros
con `validation` y evaluar una sola vez con `test` (matriz de confusión y métricas
por clase, incluida `unknown`). El agrupamiento crea desbalance: decidir pesos
o muestreo usando exclusivamente entrenamiento y reportar macro-F1 y recall
por clase además de exactitud. No se han descartado palabras para balancear.
Si se añade aumento de datos, aplicarlo únicamente a entrenamiento.
"""
display(Markdown(summary))
(RUN_DIR / "summary.md").write_text(summary, encoding="utf-8")
